# RAG Playground

Use this notebook to quickly test and run the RAG pipeline. It handles document loading, vector storage (local or remote), and answering questions using Groq's LLMs.

### How this works
- **Efficient**: It only processes new or changed files, so indexing is fast.
- **Modular**: All the logic is kept in the `src/` folder, making it easy to manage.
- **Flexible**: You can easily swap between different databases like Chroma or Typesense.

## 1. Setup and imports

In [4]:
from typing import Literal
from src.config import AppConfig, get_backend_from_env           # Defined in src/config.py
from src.embeddings import EmbeddingManager                      # Defined in src/embeddings.py
from src.ingestion import load_text_and_pdfs, split_documents    # Defined in src/ingestion.py
from src.pipeline import AdvancedRAGPipeline, build_groq_llm     # Defined in src/pipeline.py
from src.retrieval import ChromaRAGRetriever, TypesenseRAGRetriever # Defined in src/retrieval.py
from src.vectorstores import ChromaVectorStore, TypesenseVectorStore # Defined in src/vectorstores.py

# Initialize configuration
cfg = AppConfig()
print(f"Data Directory: {cfg.paths.data_dir}")
backend = get_backend_from_env(default='chroma')
print(f"Active Vector Backend: {backend}")

Data Directory: C:\Users\owner\Desktop\python\RAG\rag-1\data
Active Vector Backend: chroma


## 2. Load and index documents

This part scans your data folder, breaks files into smaller chunks, and adds them to the vector database. It uses a hashing system to avoid indexing the same content multiple times.

In [5]:
# Load the embedding model
embed_manager = EmbeddingManager()
embed_manager.load_model()

# Setup the database
chroma_store = ChromaVectorStore()
print(f"Initial record count in vector store: {chroma_store.count()}")

# Load and chunk files
raw_docs = load_text_and_pdfs(cfg.paths.data_dir)
chunks = split_documents(raw_docs)

# Create embeddings and update the store
texts = [doc.page_content for doc in chunks]
embeddings = embed_manager.generate_embeddings(texts)

# Add only new/changed documents
chroma_store.add_documents(chunks, embeddings)

print(f"Final record count in vector store: {chroma_store.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1664.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initial record count in vector store: 77
Final record count in vector store: 77


## 3. Ask a question

Now we search the database for relevant info and use the LLM to write an answer based on what it finds.

In [ ]:
# Setup the full pipeline
retriever = ChromaRAGRetriever(chroma_store, embed_manager)
llm = build_groq_llm()
pipeline = AdvancedRAGPipeline(retriever=retriever, llm=llm)

# Input your own question here based on the data you've indexed
query = "How do I use this RAG system?"
result = pipeline.query(query, top_k=3, min_score=0.1, summarize=True)

print(f"\nQUERY: {result['question']}")
print(f"\nFINAL ANSWER:\n{result['answer']}")
if result['summary']:
    print(f"\nSUMMARY: {result['summary']}")